In [4]:
import pandas as pd
import numpy as np
import re
import spacy

In [5]:
import pandas as pd

df = pd.read_json(
    "../datasets/resumes/resumes.csv",
    lines=True
)

In [6]:
df.head()

,ResumeID,Category,Name,Email,Phone,Location,Summary,Skills,Experience,Education,Text,Source
0,REAL_0001,Java Developer,Chad Griffin,contact@email.com,94105 555 4321000 10 ...,"City, State",jessica claire montgomery street san francisco...,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...,ResumeAtlas
1,REAL_0002,Java Developer,Melinda Thomas,contact@email.com,17994568777 2017 2018 20152016 3 ...,"City, State",jared arthur maica java developer 17994568777 ...,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...,ResumeAtlas
2,REAL_0003,Java Developer,Shannon Mccarthy,contact@email.com,9 555 4321000 94105 8 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas
3,REAL_0004,Java Developer,Christine Kelley,contact@email.com,9 555 4321000 94105 5 ...,"City, State",jessica claire 9 resumesampleexamplecom 555 43...,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas
4,REAL_0005,Java Developer,Karen Holt,contact@email.com,100 10 4321000 ...,"City, State",jessica claire 100 montgomery st 10th floor xx...,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,ResumeAtlas


In [7]:
df.shape

(3500, 12)

In [8]:
df.columns.tolist()

['ResumeID',
 'Category',
 'Name',
 'Email',
 'Phone',
 'Location',
 'Summary',
 'Skills',
 'Experience',
 'Education',
 'Text',
 'Source']

In [9]:
df.isnull().sum()

ResumeID      0
Category      0
Name          0
Email         0
Phone         0
Location      0
Summary       0
Skills        0
Experience    0
Education     0
Text          0
Source        0
dtype: int64

In [10]:
resume_df = df[[
    "ResumeID",
    "Category",
    "Skills",
    "Experience",
    "Education",
    "Text"
]].copy()

In [11]:
resume_df.head()

,ResumeID,Category,Skills,Experience,Education,Text
0,REAL_0001,Java Developer,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...
1,REAL_0002,Java Developer,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...
2,REAL_0003,Java Developer,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...
3,REAL_0004,Java Developer,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...
4,REAL_0005,Java Developer,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...


In [12]:
resume_df.isnull().sum()

ResumeID      0
Category      0
Skills        0
Experience    0
Education     0
Text          0
dtype: int64

In [13]:
resume_df = resume_df.drop_duplicates()

In [14]:
resume_df.shape

(3500, 6)

In [15]:
import re

In [16]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+"," ",text)

    text = re.sub(r"[^a-zA-Z ]"," ",text)

    text = re.sub(r"\s+"," ",text)

    return text.strip()

In [17]:
resume_df["clean_text"] = (
    resume_df["Text"]
    .apply(clean_text)
)

In [18]:
resume_df[
    ["Text","clean_text"]
].head()

,Text,clean_text
0,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...
1,jared arthur maica java developer 17994568777 ...,jared arthur maica java developer linkedincomi...
2,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire resumesampleexamplecom montgome...
3,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire resumesampleexamplecom montgome...
4,jessica claire 100 montgomery st 10th floor xx...,jessica claire montgomery st th floor xxx resu...


In [19]:
resume_df.to_csv(
    "../datasets/processed/processed_resumes.csv",
    index=False
)

In [20]:
resume_df.shape

(3500, 7)

In [21]:
import spacy

In [22]:
nlp = spacy.load("en_core_web_sm")

In [23]:
sample = resume_df["clean_text"].iloc[0]

doc = nlp(sample)

tokens = [
    token.lemma_
    for token in doc
    if not token.is_stop
]

tokens[:30]

['jessica',
 'claire',
 'montgomery',
 'street',
 'san',
 'francisco',
 'resumesampleexamplecom',
 'professional',
 'summary',
 'highly',
 'skilled',
 'software',
 'development',
 'professional',
 'bringing',
 'year',
 'software',
 'design',
 'development',
 'integration',
 'advanced',
 'knowledge',
 'java',
 'skill',
 'agile',
 'html',
 'xml',
 'jdbc',
 'tomcat',
 'work']

In [24]:
def preprocess(text):

    doc = nlp(text)

    words = []

    for token in doc:

        if (
            not token.is_stop
            and not token.is_punct
        ):

            words.append(
                token.lemma_
            )

    return " ".join(words)

In [25]:
resume_df["processed_text"] = (
    resume_df["clean_text"]
    .apply(preprocess)
)

In [26]:
resume_df[
    ["clean_text","processed_text"]
].head()

,clean_text,processed_text
0,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...
1,jared arthur maica java developer linkedincomi...,jared arthur maica java developer linkedincomi...
2,jessica claire resumesampleexamplecom montgome...,jessica claire resumesampleexamplecom montgome...
3,jessica claire resumesampleexamplecom montgome...,jessica claire resumesampleexamplecom montgome...
4,jessica claire montgomery st th floor xxx resu...,jessica claire montgomery st th floor xxx resu...


In [27]:
resume_df.to_csv(
    "../datasets/processed/processed_resumes.csv",
    index=False
)

In [28]:
resume_df["resume_length"] = (
    resume_df["processed_text"]
    .apply(len)
)

In [29]:
resume_df[
["Category","resume_length"]
].head()

,Category,resume_length
0,Java Developer,1425
1,Java Developer,1556
2,Java Developer,4955
3,Java Developer,11807
4,Java Developer,3871


In [30]:
resume_df.to_csv(
    "processed_resumes.csv",
    index=False
)

print("Processed dataset saved successfully")

Processed dataset saved successfully


In [31]:
resume_df.head()

,ResumeID,Category,Skills,Experience,Education,Text,clean_text,processed_text,resume_length
0,REAL_0001,Java Developer,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,1425
1,REAL_0002,Java Developer,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...,jared arthur maica java developer linkedincomi...,jared arthur maica java developer linkedincomi...,1556
2,REAL_0003,Java Developer,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire resumesampleexamplecom montgome...,jessica claire resumesampleexamplecom montgome...,4955
3,REAL_0004,Java Developer,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire resumesampleexamplecom montgome...,jessica claire resumesampleexamplecom montgome...,11807
4,REAL_0005,Java Developer,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,jessica claire montgomery st th floor xxx resu...,jessica claire montgomery st th floor xxx resu...,3871


In [32]:
resume_df.columns

Index(['ResumeID', 'Category', 'Skills', 'Experience', 'Education', 'Text',
       'clean_text', 'processed_text', 'resume_length'],
      dtype='object')

In [33]:
print("processed_text exists:", "processed_text" in resume_df.columns)
print("resume_length exists:", "resume_length" in resume_df.columns)

processed_text exists: True
resume_length exists: True


In [34]:
resume_df.isnull().sum()

ResumeID          0
Category          0
Skills            0
Experience        0
Education         0
Text              0
clean_text        0
processed_text    0
resume_length     0
dtype: int64

In [35]:
resume_df[
["Category","processed_text"]
].sample(5)

,Category,processed_text
1934,Network Security Engineer,jessica claire resumesampleexamplecom montgome...
2522,Full Stack Developer,frank kline frank kline hotmail com x port kim...
1625,DotNet Developer,jessica claire montgomery street san francisco...
1177,Testing,contact elysexparkergmailcom summary elyse par...
2346,Software Developer,james collins email james collins hotmail com ...


In [36]:
resume_df["resume_length"].describe()

count     3500.000000
mean      3021.417429
std       2791.912700
min        193.000000
25%       1084.000000
50%       1878.500000
75%       4384.000000
max      51451.000000
Name: resume_length, dtype: float64

In [37]:
skills = [
"python",
"sql",
"excel",
"power bi",
"tableau",
"machine learning",
"deep learning",
"java",
"c++",
"aws",
"azure",
"spark",
"hadoop",
"pandas",
"numpy"
]

In [38]:
def extract_skills(text):
    found = []

    text = text.lower()

    for skill in skills:
        if skill in text:
            found.append(skill)

    return found

In [39]:
resume_df["skills"] = (
    resume_df["processed_text"]
    .apply(extract_skills)
)

resume_df[
["Category","skills"]
].head()

,Category,skills
0,Java Developer,"[excel, java]"
1,Java Developer,[java]
2,Java Developer,"[python, sql, java, aws, azure]"
3,Java Developer,"[python, sql, excel, java]"
4,Java Developer,[java]


In [40]:
resume_df["skill_score"] = (
    resume_df["skills"]
    .apply(len)
)

resume_df[
["Category",
 "skill_score"]
].head()

,Category,skill_score
0,Java Developer,2
1,Java Developer,1
2,Java Developer,5
3,Java Developer,4
4,Java Developer,1


In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
job_description = """
Looking for Data Analyst.

Skills:
Python
SQL
Power BI
Excel
Dashboarding
Data Visualization
Statistics
Business Analytics
"""

In [44]:
documents = (
    [job_description] +
    resume_df["processed_text"].tolist()
)

In [45]:
vectorizer = TfidfVectorizer()

tfidf_matrix = (
    vectorizer
    .fit_transform(documents)
)

In [46]:
similarity_scores = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:]
)[0]

In [47]:
resume_df["match_score"] = (
    similarity_scores * 100
).round(2)

In [49]:
resume_df[
["Category","processed_text"]
].head()

,Category,processed_text
0,Java Developer,jessica claire montgomery street san francisco...
1,Java Developer,jared arthur maica java developer linkedincomi...
2,Java Developer,jessica claire resumesampleexamplecom montgome...
3,Java Developer,jessica claire resumesampleexamplecom montgome...
4,Java Developer,jessica claire montgomery st th floor xxx resu...


In [50]:
resume_df[
[
"Category",
"match_score"
]
].head()

,Category,match_score
0,Java Developer,2.36
1,Java Developer,0.00
2,Java Developer,1.54
3,Java Developer,3.55
4,Java Developer,0.18


In [51]:
top_candidates = (
    resume_df
    .sort_values(
        by="match_score",
        ascending=False
    )
)

top_candidates[
[
"Category",
"match_score",
"skills"
]
].head(10)

,Category,match_score,skills
946,SQL Developer,26.57,"[sql, excel, power bi, tableau, azure]"
814,SQL Developer,21.93,"[sql, excel, power bi, java]"
782,SQL Developer,20.16,"[sql, excel, power bi, java]"
900,SQL Developer,20.12,"[sql, excel, power bi, java]"
959,SQL Developer,20.12,"[sql, excel, power bi, java]"
838,SQL Developer,19.01,"[python, sql, excel, power bi, tableau, java, ..."
832,SQL Developer,18.71,"[sql, excel, power bi, azure]"
259,Python Developer,18.05,"[sql, excel, power bi, tableau, azure]"
810,SQL Developer,15.18,"[python, sql, excel, power bi]"
962,Database,15.18,"[python, sql, excel, power bi]"


In [52]:
resume_df["candidate_rank"] = (
    resume_df["match_score"]
    .rank(
        ascending=False,
        method="dense"
    )
)

In [53]:
resume_df[
[
"Category",
"match_score",
"candidate_rank"
]
].head()

,Category,match_score,candidate_rank
0,Java Developer,2.36,429.0
1,Java Developer,0.00,659.0
2,Java Developer,1.54,509.0
3,Java Developer,3.55,316.0
4,Java Developer,0.18,645.0


In [54]:
resume_df["skill_count"] = (
    resume_df["skills"]
    .apply(len)
)

In [55]:
resume_df[
[
"skills",
"skill_count"
]
].head()

,skills,skill_count
0,"[excel, java]",2
1,[java],1
2,"[python, sql, java, aws, azure]",5
3,"[python, sql, excel, java]",4
4,[java],1


In [56]:
def candidate_status(score):

    if score >= 70:
        return "Shortlisted"

    elif score >= 40:
        return "Review"

    else:
        return "Rejected"

In [57]:
resume_df["candidate_status"] = (
    resume_df["match_score"]
    .apply(candidate_status)
)

In [58]:
resume_df[
[
"match_score",
"candidate_status"
]
].head()

,match_score,candidate_status
0,2.36,Rejected
1,0.00,Rejected
2,1.54,Rejected
3,3.55,Rejected
4,0.18,Rejected


In [59]:
resume_df.to_csv(
    "resume_screening_dashboard.csv",
    index=False
)

print("Dashboard dataset created")

Dashboard dataset created


In [61]:
resume_df.to_csv(
    "resume_screening_dashboard.csv",
    index=False
)

In [62]:
resume_df.columns

Index(['ResumeID', 'Category', 'Skills', 'Experience', 'Education', 'Text',
       'clean_text', 'processed_text', 'resume_length', 'skills',
       'skill_score', 'match_score', 'candidate_rank', 'skill_count',
       'candidate_status'],
      dtype='object')

In [64]:
import os

print(os.getcwd())

c:\Users\sathwika\OneDrive\Documents\Desktop\AI_Resume_Screening_System\AI_Resume_Screening_System\notebooks


In [65]:
import sqlite3

In [66]:
conn = sqlite3.connect(
    "resume_screening.db"
)

print("Database connected")

Database connected


In [68]:
print(resume_df.columns.tolist())

['ResumeID', 'Category', 'Skills', 'Experience', 'Education', 'Text', 'clean_text', 'processed_text', 'resume_length', 'skills', 'skill_score', 'match_score', 'candidate_rank', 'skill_count', 'candidate_status']


In [69]:
resume_df = resume_df.loc[
    :,
    ~resume_df.columns.duplicated()
]

In [70]:
print(resume_df.columns.tolist())

['ResumeID', 'Category', 'Skills', 'Experience', 'Education', 'Text', 'clean_text', 'processed_text', 'resume_length', 'skills', 'skill_score', 'match_score', 'candidate_rank', 'skill_count', 'candidate_status']


In [71]:
resume_df = resume_df.rename(
    columns={
        "skills": "extracted_skills"
    }
)

In [72]:
print(resume_df.columns.tolist())

['ResumeID', 'Category', 'Skills', 'Experience', 'Education', 'Text', 'clean_text', 'processed_text', 'resume_length', 'extracted_skills', 'skill_score', 'match_score', 'candidate_rank', 'skill_count', 'candidate_status']


In [73]:
resume_df["extracted_skills"] = (
    resume_df["extracted_skills"]
    .astype(str)
)

In [74]:
resume_df.to_sql(
    "candidate_scores",
    conn,
    if_exists="replace",
    index=False
)

print("SQL table created")

SQL table created


In [75]:
query = """
SELECT *
FROM candidate_scores
LIMIT 5
"""

pd.read_sql(
    query,
    conn
)

,ResumeID,Category,Skills,Experience,Education,Text,clean_text,processed_text,resume_length,extracted_skills,skill_score,match_score,candidate_rank,skill_count,candidate_status
0,REAL_0001,Java Developer,"Python, SQL, Git, Linux",jessica claire montgomery street san francisco...,Computer Science degree,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,1425,"['excel', 'java']",2,2.36,429.0,2,Rejected
1,REAL_0002,Java Developer,"Python, SQL, Git, Linux",jared arthur maica java developer 17994568777 ...,Computer Science degree,jared arthur maica java developer 17994568777 ...,jared arthur maica java developer linkedincomi...,jared arthur maica java developer linkedincomi...,1556,['java'],1,0.00,659.0,1,Rejected
2,REAL_0003,Java Developer,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire resumesampleexamplecom montgome...,jessica claire resumesampleexamplecom montgome...,4955,"['python', 'sql', 'java', 'aws', 'azure']",5,1.54,509.0,5,Rejected
3,REAL_0004,Java Developer,"Python, SQL, Git, Linux",jessica claire 9 resumesampleexamplecom 555 43...,Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire resumesampleexamplecom montgome...,jessica claire resumesampleexamplecom montgome...,11807,"['python', 'sql', 'excel', 'java']",4,3.55,316.0,4,Rejected
4,REAL_0005,Java Developer,"Python, SQL, Git, Linux",jessica claire 100 montgomery st 10th floor xx...,Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,jessica claire montgomery st th floor xxx resu...,jessica claire montgomery st th floor xxx resu...,3871,['java'],1,0.18,645.0,1,Rejected


In [76]:
query = """
SELECT
candidate_status,
COUNT(*) AS total_candidates
FROM candidate_scores
GROUP BY candidate_status
ORDER BY total_candidates DESC
"""

pd.read_sql(query, conn)

,candidate_status,total_candidates
0,Rejected,3500


In [77]:
query = """
SELECT
ROUND(
AVG(match_score),
2
) AS average_match_score
FROM candidate_scores
"""

pd.read_sql(query, conn)

,average_match_score
0,1.84


In [78]:
query = """
SELECT
Category,
ROUND(
AVG(match_score),
2
) AS avg_match,
COUNT(*) AS candidates
FROM candidate_scores
GROUP BY Category
ORDER BY avg_match DESC
"""

pd.read_sql(query, conn)

,Category,avg_match,candidates
0,SQL Developer,4.86,180
1,Business Analyst,4.82,150
2,ETL Developer,4.54,120
3,Cybersecurity Analyst,4.05,66
4,Python Developer,3.50,200
5,Data Science,2.92,200
6,Database,2.48,150
7,Engineering Manager,1.68,30
8,DotNet Developer,1.59,140
9,React Developer,1.55,150


In [79]:
query = """
SELECT
ResumeID,
Category,
match_score,
candidate_rank,
candidate_status
FROM candidate_scores
ORDER BY match_score DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,ResumeID,Category,match_score,candidate_rank,candidate_status
0,REAL_0947,SQL Developer,26.57,1.0,Rejected
1,REAL_0815,SQL Developer,21.93,2.0,Rejected
2,REAL_0783,SQL Developer,20.16,3.0,Rejected
3,REAL_0901,SQL Developer,20.12,4.0,Rejected
4,REAL_0960,SQL Developer,20.12,4.0,Rejected
5,REAL_0839,SQL Developer,19.01,5.0,Rejected
6,REAL_0833,SQL Developer,18.71,6.0,Rejected
7,REAL_0260,Python Developer,18.05,7.0,Rejected
8,REAL_0811,SQL Developer,15.18,8.0,Rejected
9,REAL_0963,Database,15.18,8.0,Rejected


In [80]:
query = """
SELECT
Category,
ROUND(
AVG(resume_length),
0
) AS avg_resume_length
FROM candidate_scores
GROUP BY Category
ORDER BY avg_resume_length DESC
"""

pd.read_sql(query, conn)

,Category,avg_resume_length
0,React Developer,5342.0
1,ETL Developer,4895.0
2,Java Developer,4596.0
3,Python Developer,4290.0
4,SQL Developer,4243.0
5,Network Security Engineer,4180.0
6,SAP Developer,4086.0
7,DotNet Developer,3844.0
8,Database,3838.0
9,DevOps,3763.0


In [81]:
query = """
SELECT
Category,
ROUND(
AVG(skill_score),
2
) AS avg_skill_score
FROM candidate_scores
GROUP BY Category
ORDER BY avg_skill_score DESC
"""

pd.read_sql(query, conn)

,Category,avg_skill_score
0,Python Developer,4.37
1,Technical Writer,4.00
2,Technical Lead,4.00
3,System Administrator,4.00
4,Product Manager,4.00
5,Principal Engineer,4.00
6,Engineering Manager,4.00
7,AI Engineer,4.00
8,Backend Developer,3.57
9,Software Developer,3.56


In [83]:
import os

os.makedirs(
    "data",
    exist_ok=True
)

print("data folder created")

data folder created


In [84]:
final_df = pd.read_sql(
"""
SELECT *
FROM candidate_scores
""",
conn
)

final_df.to_csv(
"data/resume_screening_final.csv",
index=False
)

print("Final Tableau dataset ready")

Final Tableau dataset ready


In [85]:
resume_df["extracted_skills"] = (
    resume_df["extracted_skills"]
    .astype(str)
)

resume_df.to_csv(
    "data/resume_screening_final.csv",
    index=False
)

In [86]:
print(resume_df.columns)

Index(['ResumeID', 'Category', 'Skills', 'Experience', 'Education', 'Text',
       'clean_text', 'processed_text', 'resume_length', 'extracted_skills',
       'skill_score', 'match_score', 'candidate_rank', 'skill_count',
       'candidate_status'],
      dtype='object')


In [87]:
resume_df.columns = [
    col.replace(" ","_")
    for col in resume_df.columns
]

In [88]:
resume_df.to_csv(
    "data/resume_screening_final.csv",
    index=False,
    encoding="utf-8"
)

In [89]:
print(resume_df["resume_length"].dtype)

int64


In [90]:
resume_df["resume_length"] = (
    resume_df["resume_length"]
    .astype("int64")
)

In [91]:
print(
resume_df["resume_length"].dtype
)

int64


In [92]:
resume_df.to_csv(
    "data/resume_screening_final.csv",
    index=False,
    encoding="utf-8"
)

In [93]:
import os

os.makedirs(
    "data",
    exist_ok=True
)

final_df = pd.read_sql(
"""
SELECT *
FROM candidate_scores
""",
conn
)

final_df.to_csv(
"data/resume_screening_final.csv",
index=False
)

print("Final Tableau dataset created")

Final Tableau dataset created


In [94]:
os.listdir("data")

['resume_screening_final.csv']